# GHG site climate extraction (Google Earth Engine)

Extracts weather and climate normals at the GHG chamber sites, one row per site visit (`Site_ID_round`).

**ERA5-Land daily** (`ECMWF/ERA5_LAND/DAILY_AGGR`, ~11 km): summarized over the **14 days before** each sample `Date`. The window is the whole days `[Date − 14 d, Date)`, so the sample day itself (a partial day at the time of sampling) is excluded.

| column | ERA5 band | summary | unit |
|---|---|---|---|
| `era5_temperature_2m_14d` | `temperature_2m` | mean of daily means | °C |
| `era5_temperature_2m_max_14d` | `temperature_2m_max` | mean of daily maxima | °C |
| `era5_temperature_2m_min_14d` | `temperature_2m_min` | mean of daily minima | °C |
| `era5_total_precipitation_sum_14d` | `total_precipitation_sum` | 14-day total | mm |
| `era5_n_days` | – | days in the window (QA, should be 14) | – |

**PRISM 1991–2020 normals** (`OREGONSTATE/PRISM/Norm91m`, 30″ ≈ 800 m): `ppt`, `tmean`, `tmax`, `tmin`, as
- **annual** normals (`prism_*_annual`): `ppt` = sum of the 12 monthly normals (mm/yr); temperatures = mean of the 12 monthly normals (°C)
- **sample-month** normals (`prism_*_month`): the normal for the calendar month of `Date` (mm/month, °C). Subtracting them from the ERA5 values gives a rough anomaly, but note the two products differ in resolution (11 km vs 800 m).

The collection holds two copies of the monthly normals: `01`–`12` (dataset version M3) and `202001`–`202012`, PRISM's later revision (M4/M5, `normals/9120`). This notebook uses the later revision.

Output: `Data/GHG/NYDEC_GHG_Site_Climate.csv`, joined to the GHG data by `Site_ID_round`.

**Environment:** a Python env with `earthengine-api`, `geopandas`, `pandas`, e.g. `/workdir/ajs544/miniconda3/envs/gee-env` or `.../geospatial-processing`. Authentication uses the service-account key in `NYS_Wetlands_Data/Python_Code_Analysis/API_Key/` (works non-interactively on BioHPC). If no key works, it falls back to `earthengine authenticate` user credentials.

In [1]:
from pathlib import Path
import glob

import ee
import geopandas as gpd
import pandas as pd

PROJECT_ROOT = Path("/ibstorage/anthony/NYS_Wetlands_GHG")
SITES = PROJECT_ROOT / "Data/GHG/NYDEC_GHG_Sites_6347.gpkg"
OUT_CSV = PROJECT_ROOT / "Data/GHG/NYDEC_GHG_Site_Climate.csv"
KEY_DIR = Path("/ibstorage/anthony/NYS_Wetlands_Data/Python_Code_Analysis/API_Key")
EE_PROJECT = "earthengineajs"

WINDOW_DAYS = 14
ERA5_ID = "ECMWF/ERA5_LAND/DAILY_AGGR"
ERA5_TEMP = ["temperature_2m", "temperature_2m_max", "temperature_2m_min"]
ERA5_PPT = "total_precipitation_sum"
PRISM_ID = "OREGONSTATE/PRISM/Norm91m"
PRISM_BANDS = ["ppt", "tmean", "tmax", "tmin"]

## Authenticate

In [2]:
def initialize():
    """Service-account key first (non-interactive), then user credentials."""
    failures = []
    for key in sorted(glob.glob(str(KEY_DIR / "*.json"))):
        try:
            ee.Initialize(ee.ServiceAccountCredentials(None, key), project=EE_PROJECT)
            print("Earth Engine: service account", Path(key).name)
            return
        except Exception as exc:  # revoked keys / accounts without EE access
            failures.append(f"{Path(key).name}: {exc}")
    try:
        ee.Initialize(project=EE_PROJECT)
        print("Earth Engine: user credentials")
    except Exception as exc:
        raise RuntimeError("Could not initialize Earth Engine:\n" + "\n".join(failures + [str(exc)]))

initialize()

Earth Engine: service account earthengineajs-ajs_gee_access.json


## Site visits

96 rows = 48 sites × 2 rounds. Points are reprojected from EPSG:6347 to WGS84 for Earth Engine.

In [3]:
sites = gpd.read_file(SITES)[["Site_ID", "Site_ID_round", "Round_Number", "Date", "geometry"]]
sites["Date"] = pd.to_datetime(sites["Date"]).dt.strftime("%Y-%m-%d")
assert sites["Site_ID_round"].is_unique and sites["Date"].notna().all()
wgs = sites.to_crs("EPSG:4326")

fc = ee.FeatureCollection([
    ee.Feature(ee.Geometry.Point([g.x, g.y]), {"Site_ID_round": sid, "Date": d})
    for g, sid, d in zip(wgs.geometry, wgs["Site_ID_round"], wgs["Date"])
])
print(len(sites), "visits;", sites["Date"].min(), "to", sites["Date"].max())

96 visits; 2022-06-27 to 2025-09-11


ERROR 1: PROJ: proj_create_from_database: Open of /local/workdir/ajs544/miniconda3/envs/geospatial-processing/share/proj failed


## Climate images

ERA5 is built per visit (its window depends on the date). The PRISM annual image is static; the sample-month image is chosen per visit.

In [4]:
era5 = ee.ImageCollection(ERA5_ID).select(ERA5_TEMP + [ERA5_PPT])
ERA5_SCALE = era5.first().select("temperature_2m").projection().nominalScale()

# Later revision of the 1991-2020 normals: system:index 202001..202012
prism = (ee.ImageCollection(PRISM_ID)
         .filter(ee.Filter.stringStartsWith("system:index", "2020"))
         .select(PRISM_BANDS))
assert prism.size().getInfo() == 12
PRISM_SCALE = prism.first().select("tmean").projection().nominalScale()

prism_annual = (prism.select("ppt").sum()
                .addBands(prism.select(["tmean", "tmax", "tmin"]).mean())
                .rename([f"prism_{b}_annual" for b in PRISM_BANDS]))


def extract(feat):
    feat = ee.Feature(feat)
    geom = feat.geometry()
    end = ee.Date(feat.get("Date"))  # UTC midnight of the sample day
    start = end.advance(-WINDOW_DAYS, "day")
    window = era5.filterDate(start, end)

    era5_img = (window.select(ERA5_TEMP).mean().subtract(273.15)               # K -> degC
                .addBands(window.select(ERA5_PPT).sum().multiply(1000))        # m -> mm
                .rename([f"era5_{b}_{WINDOW_DAYS}d" for b in ERA5_TEMP + [ERA5_PPT]]))
    era5_vals = era5_img.reduceRegion(ee.Reducer.first(), geom, ERA5_SCALE)

    month_img = (prism.filter(ee.Filter.calendarRange(end.get("month"), end.get("month"), "month"))
                 .first().rename([f"prism_{b}_month" for b in PRISM_BANDS]))
    prism_vals = prism_annual.addBands(month_img).reduceRegion(ee.Reducer.first(), geom, PRISM_SCALE)

    return (feat.set(era5_vals).set(prism_vals)
            .set({"era5_n_days": window.size(),
                  "era5_start": start.format("YYYY-MM-dd"),
                  "era5_end_exclusive": end.format("YYYY-MM-dd")}))

## Extract

In [5]:
info = fc.map(extract).getInfo()
clim = pd.DataFrame([f["properties"] for f in info["features"]])

cols = (["Site_ID_round", "era5_start", "era5_end_exclusive", "era5_n_days"]
        + [f"era5_{b}_{WINDOW_DAYS}d" for b in ERA5_TEMP + [ERA5_PPT]]
        + [f"prism_{b}_annual" for b in PRISM_BANDS]
        + [f"prism_{b}_month" for b in PRISM_BANDS])
missing_cols = [c for c in cols if c not in clim.columns]
assert not missing_cols, f"No values returned for {missing_cols} (masked pixels?)"
clim = clim[cols]

out = sites.drop(columns="geometry").merge(clim, on="Site_ID_round", how="left", validate="1:1")
out.describe().T.round(2)

,count,mean,std,min,25%,50%,75%,max
Round_Number,96.0,1.50,0.50,1.00,1.00,1.50,2.00,2.00
era5_n_days,96.0,14.00,0.00,14.00,14.00,14.00,14.00,14.00
era5_temperature_2m_14d,96.0,19.77,1.98,15.49,18.57,19.71,21.21,23.73
era5_temperature_2m_max_14d,96.0,24.50,2.00,20.15,23.36,24.39,25.56,29.38
era5_temperature_2m_min_14d,96.0,15.67,2.19,11.25,13.98,15.76,17.41,20.01
era5_total_precipitation_sum_14d,96.0,51.45,34.16,0.98,26.23,40.53,71.15,159.15
prism_ppt_annual,96.0,1103.15,114.08,866.77,1034.67,1110.59,1174.20,1400.85
prism_tmean_annual,96.0,7.76,1.62,4.79,6.86,7.85,9.16,11.15
prism_tmax_annual,96.0,13.16,1.44,10.66,12.11,13.07,14.20,16.11
prism_tmin_annual,96.0,2.36,1.88,-1.24,1.63,2.40,3.80,6.30


## QA

ERA5-Land is land-only, so a point in a coastal or lake-edge cell can come back empty, as can a pixel PRISM masks.

In [6]:
value_cols = [c for c in cols if c.startswith(("era5_", "prism_")) and c not in ("era5_start", "era5_end_exclusive")]
na = out[value_cols].isna().sum()
print("Missing values:", "none" if na.sum() == 0 else "")
print(na[na > 0].to_string() if na.sum() else "")
assert (out["era5_n_days"] == WINDOW_DAYS).all(), "Some windows are not 14 full days (ERA5 lag?)"

# The two rounds should share normals; ERA5 differs by date
by_site = out.groupby("Site_ID")[[f"prism_{b}_annual" for b in PRISM_BANDS]].nunique()
assert (by_site == 1).all().all(), "Annual normals differ between rounds of a site"
print("OK:", len(out), "visits,", out["Site_ID"].nunique(), "sites")

Missing values: none

OK: 96 visits, 48 sites


## Save

In [7]:
out.to_csv(OUT_CSV, index=False)
print("Wrote", OUT_CSV, out.shape)

Wrote /ibstorage/anthony/NYS_Wetlands_GHG/Data/GHG/NYDEC_GHG_Site_Climate.csv (96, 19)
